In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, ttest_1samp

pd.set_option("display.float_format", "{:.4f}".format)


# ============================================================
# Build pair-level matched panel
# ============================================================

def build_matched_panel(matches, month_result):

    matches = matches.copy()
    month_result = month_result.copy()

    matches["cohort"] = pd.to_datetime(matches["adoption_month"]).dt.to_period("M")
    month_result["TIDPUNKT"] = pd.to_datetime(month_result["TIDPUNKT"]).dt.to_period("M")

    month_result["top3_mean_consumption"] = pd.to_numeric(
        month_result["top3_mean_consumption"],
        errors="coerce"
    )

    # Treated household-month panel
    treated_panel = (
        matches[["treated_id", "cohort"]]
        .drop_duplicates()
        .merge(
            month_result,
            left_on="treated_id",
            right_on="aID",
            how="left"
        )
    )

    treated_panel = treated_panel.rename(
        columns={"top3_mean_consumption": "treated_y"}
    )

    # Control household-month panel
    control_panel = (
        matches[["treated_id", "control_id", "cohort"]]
        .merge(
            month_result,
            left_on="control_id",
            right_on="aID",
            how="left"
        )
    )

    control_panel = control_panel.rename(
        columns={"top3_mean_consumption": "control_y"}
    )

    # Average matched controls for each treated household-month
    control_mean = (
        control_panel
        .groupby(["treated_id", "cohort", "TIDPUNKT"], as_index=False)
        .agg(
            matched_control_y=("control_y", "mean"),
            n_controls=("control_y", "count")
        )
    )

    # Keep useful columns from treated panel
    keep_cols = ["treated_id", "cohort", "TIDPUNKT", "treated_y"]

    if "price" in treated_panel.columns:
        keep_cols.append("price")

    df = treated_panel[keep_cols].merge(
        control_mean,
        on=["treated_id", "cohort", "TIDPUNKT"],
        how="inner"
    )

    df["event_time"] = (df["TIDPUNKT"] - df["cohort"]).apply(lambda x: x.n)

    # Pair-level effect
    df["pair_effect"] = df["treated_y"] - df["matched_control_y"]

    # Compatibility with old function names
    df["top3_mean_consumption"] = df["pair_effect"]
    df["aID"] = df["treated_id"]
    df["treatment"] = 1

    return df


# ============================================================
# Pair-level dynamic effect
# ============================================================

def compute_effect(df, outcome_col="top3_mean_consumption"):

    results = []

    for t in sorted(df["event_time"].dropna().unique()):

        d = df[df["event_time"] == t].copy()
        d = d.dropna(subset=["pair_effect"])

        if len(d) < 2:
            continue

        effect = d["pair_effect"].mean()
        se = d["pair_effect"].std(ddof=1) / np.sqrt(len(d))

        t_stat = effect / se if se > 0 else np.nan
        p_value = 2 * (1 - norm.cdf(abs(t_stat))) if se > 0 else np.nan

        results.append({
            "event_time": t,
            "mean_treated": d["treated_y"].mean(),
            "mean_control": d["matched_control_y"].mean(),
            "effect": effect,
            "se": se,
            "t_stat": t_stat,
            "p_value": p_value,
            "n_treated": len(d),
            "n_control": d["n_controls"].sum()
        })

    return pd.DataFrame(results)


def compute_effect_by_cohort(df, outcome_col="top3_mean_consumption"):

    results = []

    for c in sorted(df["cohort"].dropna().unique()):

        d_cohort = df[df["cohort"] == c].copy()

        for t in sorted(d_cohort["event_time"].dropna().unique()):

            d = d_cohort[d_cohort["event_time"] == t].copy()
            d = d.dropna(subset=["pair_effect"])

            if len(d) < 2:
                continue

            effect = d["pair_effect"].mean()
            se = d["pair_effect"].std(ddof=1) / np.sqrt(len(d))

            t_stat = effect / se if se > 0 else np.nan
            p_value = 2 * (1 - norm.cdf(abs(t_stat))) if se > 0 else np.nan

            results.append({
                "cohort": c,
                "event_time": t,
                "mean_treated": d["treated_y"].mean(),
                "mean_control": d["matched_control_y"].mean(),
                "effect": effect,
                "se": se,
                "t_stat": t_stat,
                "p_value": p_value,
                "n_treated": len(d),
                "n_control": d["n_controls"].sum()
            })

    return pd.DataFrame(results)


# ============================================================
# Confidence interval
# ============================================================

def add_confidence_interval(effect_df, alpha=0.05):

    effect_df = effect_df.copy()

    z = norm.ppf(1 - alpha / 2)

    effect_df["ci_low"] = effect_df["effect"] - z * effect_df["se"]
    effect_df["ci_high"] = effect_df["effect"] + z * effect_df["se"]

    return effect_df


# ============================================================
# Pre-trend test
# ============================================================

def pretrend_test(effect_df, pre_start=-12, pre_end=-1):

    pre = effect_df[
        (effect_df["event_time"] >= pre_start) &
        (effect_df["event_time"] <= pre_end)
    ].copy()

    pre = pre.dropna(subset=["effect"])

    if len(pre) < 2:
        return {
            "pre_start": pre_start,
            "pre_end": pre_end,
            "n_periods": len(pre),
            "mean_pre_effect": np.nan,
            "t_stat": np.nan,
            "p_value": np.nan
        }

    test = ttest_1samp(pre["effect"], popmean=0)

    return {
        "pre_start": pre_start,
        "pre_end": pre_end,
        "n_periods": len(pre),
        "mean_pre_effect": pre["effect"].mean(),
        "t_stat": test.statistic,
        "p_value": test.pvalue
    }


# ============================================================
# Plots
# ============================================================

def plot_dynamic_effect(effect_df):

    plt.figure(figsize=(8, 5))

    plt.plot(effect_df["event_time"], effect_df["effect"], marker="o")

    plt.fill_between(
        effect_df["event_time"],
        effect_df["ci_low"],
        effect_df["ci_high"],
        alpha=0.2
    )

    plt.axhline(0, linestyle="--")
    plt.axvline(0, linestyle="--")

    plt.xlabel("Event Time")
    plt.ylabel("Pair-level Treatment Effect")
    plt.title("Pair-level Dynamic Treatment Effect")

    plt.show()


def plot_dynamic_by_cohort(effect_df):

    for c in effect_df["cohort"].unique():

        d = effect_df[effect_df["cohort"] == c].sort_values("event_time")

        plt.figure(figsize=(6, 4))

        plt.plot(d["event_time"], d["effect"], marker="o")

        plt.fill_between(
            d["event_time"],
            d["ci_low"],
            d["ci_high"],
            alpha=0.2
        )

        plt.axhline(0, linestyle="--")
        plt.axvline(0, linestyle="--")

        plt.title(f"Cohort = {c}")
        plt.xlabel("Event Time")
        plt.ylabel("Pair-level Treatment Effect")

        plt.show()


def plot_all_cohorts(effect_df):

    plt.figure(figsize=(8, 5))

    for c in effect_df["cohort"].unique():

        d = effect_df[effect_df["cohort"] == c].sort_values("event_time")
        plt.plot(d["event_time"], d["effect"], alpha=0.5, label=str(c))

    plt.axhline(0, linestyle="--")
    plt.axvline(0, linestyle="--")

    plt.xlabel("Event Time")
    plt.ylabel("Pair-level Treatment Effect")
    plt.title("Pair-level Dynamic Effect by Cohort")

    plt.legend(bbox_to_anchor=(1.05, 1))
    plt.show()


def plot_pair_effect_distribution(df, event_time=0, bins=40):

    d = df[df["event_time"] == event_time].copy()
    d = d.dropna(subset=["pair_effect"])

    if d.empty:
        print(f"No data for event_time = {event_time}")
        return

    plt.figure(figsize=(8, 5))

    plt.hist(
        d["pair_effect"],
        bins=bins,
        edgecolor="black",
        alpha=0.7
    )

    plt.axvline(0, linestyle="--", label="Zero effect")
    plt.axvline(d["pair_effect"].mean(), linestyle="-", label="Mean effect")

    plt.xlabel("Treated outcome - matched control mean")
    plt.ylabel("Number of treated households")
    plt.title(f"Distribution of Pair-level Effects, event_time = {event_time}")
    plt.legend()

    plt.show()


def plot_pair_effect_distribution_by_period(df, event_times=(0, 1, 2), bins=40):

    for t in event_times:
        plot_pair_effect_distribution(df, event_time=t, bins=bins)


# ============================================================
# ATT
# ============================================================

def compute_average_treatment_effect(
    effect_df,
    post_period_only=True,
    max_post_periods=None
):

    df = effect_df.copy()

    if post_period_only:
        df = df[df["event_time"] >= 0]

    if max_post_periods is not None:
        df = df[df["event_time"] < max_post_periods]

    df = df.dropna(subset=["effect"])

    if len(df) < 2:
        return {
            "ATT": np.nan,
            "SE": np.nan,
            "t_stat": np.nan,
            "p_value": np.nan,
            "CI_low": np.nan,
            "CI_high": np.nan,
            "n_periods": len(df)
        }

    att = df["effect"].mean()
    se = df["effect"].std(ddof=1) / np.sqrt(len(df))

    t_stat = att / se if se > 0 else np.nan
    p_value = 2 * (1 - norm.cdf(abs(t_stat))) if se > 0 else np.nan

    ci_low = att - 1.96 * se
    ci_high = att + 1.96 * se

    return {
        "ATT": att,
        "SE": se,
        "t_stat": t_stat,
        "p_value": p_value,
        "CI_low": ci_low,
        "CI_high": ci_high,
        "n_periods": len(df)
    }


def compute_att_by_cohort(
    effect_df,
    post_period_only=True,
    max_post_periods=None
):

    results = []

    for c in effect_df["cohort"].unique():

        d = effect_df[effect_df["cohort"] == c].copy()

        if post_period_only:
            d = d[d["event_time"] >= 0]

        if max_post_periods is not None:
            d = d[d["event_time"] < max_post_periods]

        d = d.dropna(subset=["effect"])

        if len(d) < 2:
            continue

        att = d["effect"].mean()
        se = d["effect"].std(ddof=1) / np.sqrt(len(d))

        t_stat = att / se if se > 0 else np.nan
        p_value = 2 * (1 - norm.cdf(abs(t_stat))) if se > 0 else np.nan

        ci_low = att - 1.96 * se
        ci_high = att + 1.96 * se

        results.append({
            "cohort": c,
            "ATT": att,
            "SE": se,
            "t_stat": t_stat,
            "p_value": p_value,
            "CI_low": ci_low,
            "CI_high": ci_high,
            "n_periods": len(d)
        })

    return pd.DataFrame(results)


# ============================================================
# Full analysis
# ============================================================

def run_full_analysis(
    df,
    outcome_col="top3_mean_consumption",
    pre_start=-12,
    pre_end=-1
):

    print("===== OVERALL PAIR-LEVEL EFFECT =====")

    overall_df = compute_effect(df, outcome_col)
    overall_df = add_confidence_interval(overall_df)

    print("\nOverall pair-level dynamic effect:")
    print(overall_df.round(4))

    plot_dynamic_effect(overall_df)

    print("\n===== PAIR EFFECT DISTRIBUTION =====")
    plot_pair_effect_distribution_by_period(
        df,
        event_times=(-1, 0, 1, 2),
        bins=40
    )

    print("\n===== PRE-TREND TEST =====")
    pretrend = pretrend_test(overall_df, pre_start=pre_start, pre_end=pre_end)
    print(pretrend)

    print("\n===== OVERALL ATT: ALL POST PERIODS =====")
    att_all = compute_average_treatment_effect(
        overall_df,
        post_period_only=True,
        max_post_periods=None
    )
    print(att_all)

    print("\n===== OVERALL ATT: FIRST 3 POST PERIODS =====")
    att_3m = compute_average_treatment_effect(
        overall_df,
        post_period_only=True,
        max_post_periods=3
    )
    print(att_3m)

    print("\n===== COHORT PAIR-LEVEL DYNAMIC EFFECT =====")

    cohort_df = compute_effect_by_cohort(df, outcome_col)
    cohort_df = add_confidence_interval(cohort_df)

    print("\nCohort pair-level dynamic effect:")
    print(cohort_df.round(4))

    plot_dynamic_by_cohort(cohort_df)
    plot_all_cohorts(cohort_df)

    print("\n===== ATT BY COHORT: ALL POST PERIODS =====")
    att_cohort_all_df = compute_att_by_cohort(
        cohort_df,
        post_period_only=True,
        max_post_periods=None
    )
    print(att_cohort_all_df.round(4))

    print("\n===== ATT BY COHORT: FIRST 3 POST PERIODS =====")
    att_cohort_3m_df = compute_att_by_cohort(
        cohort_df,
        post_period_only=True,
        max_post_periods=3
    )
    print(att_cohort_3m_df.round(4))

    return {
        "overall": overall_df,
        "cohort": cohort_df,
        "att_all": att_all,
        "att_3m": att_3m,
        "att_by_cohort_all": att_cohort_all_df,
        "att_by_cohort_3m": att_cohort_3m_df,
        "pretrend": pretrend
    }


# ============================================================
# Save results
# ============================================================

def save_results(results, save_path):

    os.makedirs(save_path, exist_ok=True)

    if "overall" in results:
        results["overall"].to_csv(
            os.path.join(save_path, "overall_pair_dynamic.csv"),
            index=False
        )

    if "cohort" in results:
        results["cohort"].to_csv(
            os.path.join(save_path, "cohort_pair_dynamic.csv"),
            index=False
        )

    if "pretrend" in results:
        pd.DataFrame([results["pretrend"]]).to_csv(
            os.path.join(save_path, "pretrend_test.csv"),
            index=False
        )

    if "att_all" in results:
        pd.DataFrame([results["att_all"]]).to_csv(
            os.path.join(save_path, "att_overall_all_post.csv"),
            index=False
        )

    if "att_3m" in results:
        pd.DataFrame([results["att_3m"]]).to_csv(
            os.path.join(save_path, "att_overall_first_3m.csv"),
            index=False
        )

    if "att_by_cohort_all" in results:
        results["att_by_cohort_all"].to_csv(
            os.path.join(save_path, "att_by_cohort_all_post.csv"),
            index=False
        )

    if "att_by_cohort_3m" in results:
        results["att_by_cohort_3m"].to_csv(
            os.path.join(save_path, "att_by_cohort_first_3m.csv"),
            index=False
        )

    print(f"✅ Results saved to: {save_path}")

